# Chapter 01. 베스트셀러 데이터 이해와 기본 전처리

교보문고 온라인 베스트셀러 Excel 데이터를 텍스트 분석에 쓸 수 있도록 기본 전처리를 수행한다.

**진행 순서**

Excel 불러오기 -> 크기/컬럼 확인 -> 필요한 컬럼 선택 -> 컬럼명 정리 -> 결측치 확인/처리
-> 판매가 숫자형 변환 -> 발행일 날짜형 변환 -> 문자열 공백 정리 -> 중복 확인 -> 최종 검증 -> CSV 저장

**각 실습의 진행 방식**

해야 할 일 이해 -> 인공지능에게 프롬프트 작성 -> 코드 초안 확인 -> 직접 실행 -> 결과 확인 -> 필요한 부분 수정 -> Markdown 정리

**주의:** 인공지능이 작성한 코드는 정답이 아니다.
코드가 오류 없이 실행되는 것과 결과가 맞는 것은 다르다. 각 단계마다 출력값을 직접 읽어본다.

---
# 실습 1. Excel 파일 불러오기

## 해야 할 일

pandas를 이용해 Excel 파일을 DataFrame으로 불러온다.
Notebook과 Excel 파일은 **같은 폴더**에 둔다.

## 프롬프트 작성 가이드

프롬프트에는 다음 내용을 포함한다.

- Python 초보자라는 점
- pandas를 사용할 것
- Excel 파일명이 무엇인지
- DataFrame 변수명을 무엇으로 사용할지
- 코드를 복잡하게 작성하지 말 것
- 불러온 뒤 앞의 5행을 확인할 것

## 프롬프트 예시

```
Python과 pandas를 처음 배우고 있습니다.
현재 Notebook과 같은 폴더에
'교보문고_온라인_베스트셀러_상품리스트.xlsx' 파일이 있습니다.
pandas를 이용해 이 파일을 df라는 DataFrame으로 불러오고,
앞의 5행을 확인하는 가장 간단한 코드를 작성해 주세요.
초보자가 이해하기 쉽도록 불필요하게 복잡한 코드는 사용하지 말고,
각 코드가 무엇을 하는지 짧게 설명해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

### 먼저 작업 폴더를 맞춘다

이 저장소의 `.vscode/settings.json`에 `"jupyter.notebookFileRoot": "${workspaceFolder}"`가 있어서,
Notebook을 실행하면 작업 폴더가 노트북이 있는 곳이 아니라 **저장소 루트**가 된다.
그래서 파일 이름만 적으면 `FileNotFoundError`가 난다.
아래 셀에서 작업 폴더를 `book-text-ml`로 맞춘 뒤 진행한다.

In [1]:
import os
from pathlib import Path

# 노트북이 있는 book-text-ml 폴더를 찾아 그쪽으로 이동한다.
# 폴더가 저장소 루트에 있든 notebooks 안에 있든 모두 찾는다.
if Path.cwd().name != "book-text-ml":
    for candidate in [Path("book-text-ml"), *Path(".").glob("*/book-text-ml")]:
        if candidate.is_dir():
            os.chdir(candidate)
            break

print("작업 폴더:", Path.cwd())

작업 폴더: C:\dev\llm-data-analysis-course\notebooks\book-text-ml


## 프롬프트 예시

```
Python과 pandas를 처음 배우고 있습니다.
현재 Notebook과 같은 폴더에
'교보문고_온라인_베스트셀러_상품리스트.xlsx' 파일이 있습니다.
pandas를 이용해 이 파일을 df라는 DataFrame으로 불러오고,
앞의 5행을 확인하는 코드를 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [2]:
import pandas as pd

# Excel 파일 불러오기 (Notebook과 같은 폴더에 있어야 함)
# 교보문고 > 베스트셀러 > 온라인 베스트 > 일간 > "엑셀로 받기"로 내려받은 파일
file_name = "교보문고_온라인_베스트셀러_상품리스트.xlsx"
df = pd.read_excel(file_name)

print(type(df))   # pandas DataFrame인지 확인
df.head()         # 앞의 5행 확인

<class 'pandas.DataFrame'>


C:\Users\apll1\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,순위,상품코드,판매상품 ID,상품명,정가,판매가,할인율,적립율,적립예정포인트,인물,출판사,발행(출시)일자,분야
0,1,9791199489561,S000220308313,"세네카, 오늘을 빼앗기고 있는 당신에게","18,000","16,200",10%,5%,900,세네카,논픽션,20260701.0,인문
1,2,9791175488854,S000221248222,흔한남매 23,"16,800","15,120",10%,5%,840,흔한남매,미래엔아이세움,20260916.0,어린이(초등)
2,3,9791124038826,S000221075112,머니 트렌드 2027,"27,500","24,750",10%,5%,"1,370",정태익 외,북모먼트,20260916.0,경제/경영
3,4,9788937460586,S000000620195,싯다르타,"8,000","7,200",10%,5%,400,헤르만 헤세,민음사,20020120.0,소설
4,5,9791124137635,S000220693844,한국사 이상현상 연구원(일반판),"22,000","19,800",10%,5%,"1,100",최인서,다이브,20260831.0,소설


### 실행 후 확인할 것

- `df`가 pandas DataFrame으로 출력되는가
- 실제 도서 데이터가 표로 나오는가

### 오류가 발생한다면

`FileNotFoundError`가 나면 Notebook과 Excel 파일이 같은 폴더에 있는지 먼저 확인한다.
아래 셀로 현재 폴더의 파일 목록을 볼 수 있다.

인공지능에게 질문할 때는 **실제 오류 메시지를 그대로** 붙여넣는다.

```
아래 코드를 실행했는데 FileNotFoundError가 발생했습니다.
오류 메시지는 다음과 같습니다.
[실제 오류 메시지 붙여넣기]
초보자가 확인해야 할 내용을 순서대로 설명해 주세요.
현재 코드는 최대한 적게 수정하고 싶습니다.
```

## 프롬프트 예시

```
Notebook에서 파일을 못 찾는 오류가 났습니다.
현재 작업 폴더가 어디인지와 그 폴더에 어떤 파일이 있는지 확인하고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [3]:
# 오류가 났을 때만 실행 - 현재 폴더의 파일 목록 확인
import os
print("현재 폴더:", os.getcwd())
for f in os.listdir():
    print(f)

현재 폴더: C:\dev\llm-data-analysis-course\notebooks\book-text-ml
book_bestseller_clean.csv
book_bestseller_weekly_clean.csv
chapter01.ipynb
chapter02.ipynb
chapter02_top30_words.csv
chapter02_wordcloud.png
chapter03.ipynb
chapter03_count_matrix.npz
chapter03_count_top_terms.csv
chapter03_tfidf_matrix.npz
chapter03_tfidf_top_terms.csv
교보문고_온라인_베스트셀러_상품리스트.xlsx
교보문고_온라인_주간_베스트셀러_상품리스트.xlsx


---
# 실습 2. 데이터의 크기와 컬럼 확인하기

## 해야 할 일

Excel을 불러왔다고 바로 전처리를 시작하지 않는다.
먼저 어떤 데이터인지 확인한다.

- 행은 몇 개인가
- 컬럼은 몇 개인가
- 컬럼 이름은 무엇인가
- 각 컬럼의 자료형은 무엇인가
- 결측치는 있는가

## 프롬프트 예시

```
pandas DataFrame df를 불러온 상태입니다.
초보자가 데이터의 기본 구조를 확인하려고 합니다.
다음 내용을 각각 확인할 수 있는 간단한 pandas 코드를 작성해 주세요.
1. 전체 행과 컬럼 개수
2. 컬럼 이름 목록
3. 각 컬럼의 자료형
4. 각 컬럼의 결측치 개수
5. 앞의 5행
shape, columns, dtypes, isna(), head() 정도의 기본 기능을 사용해 주세요.
한 셀에서 실행할 수 있도록 간단하게 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [4]:
print("행, 컬럼 개수:", df.shape)
print()

print("컬럼 이름 목록:")
print(df.columns.tolist())
print()

print("각 컬럼의 자료형:")
print(df.dtypes)
print()

print("각 컬럼의 결측치 개수:")
print(df.isna().sum())

행, 컬럼 개수: (986, 13)

컬럼 이름 목록:
['순위', '상품코드', '판매상품 ID', '상품명', '정가', '판매가', '할인율', '적립율', '적립예정포인트', '인물', '출판사', '발행(출시)일자', '분야']

각 컬럼의 자료형:
순위            int64
상품코드            str
판매상품 ID         str
상품명             str
정가              str
판매가             str
할인율             str
적립율             str
적립예정포인트         str
인물              str
출판사             str
발행(출시)일자    float64
분야              str
dtype: object

각 컬럼의 결측치 개수:
순위           0
상품코드         2
판매상품 ID      0
상품명          0
정가           0
판매가          0
할인율          0
적립율          0
적립예정포인트      0
인물          10
출판사         11
발행(출시)일자     8
분야          64
dtype: int64


## 프롬프트 예시

```
pandas DataFrame df를 불러온 상태입니다.
표 형태로 앞의 5행을 보고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [5]:
df.head()

,순위,상품코드,판매상품 ID,상품명,정가,판매가,할인율,적립율,적립예정포인트,인물,출판사,발행(출시)일자,분야
0,1,9791199489561,S000220308313,"세네카, 오늘을 빼앗기고 있는 당신에게","18,000","16,200",10%,5%,900,세네카,논픽션,20260701.0,인문
1,2,9791175488854,S000221248222,흔한남매 23,"16,800","15,120",10%,5%,840,흔한남매,미래엔아이세움,20260916.0,어린이(초등)
2,3,9791124038826,S000221075112,머니 트렌드 2027,"27,500","24,750",10%,5%,"1,370",정태익 외,북모먼트,20260916.0,경제/경영
3,4,9788937460586,S000000620195,싯다르타,"8,000","7,200",10%,5%,400,헤르만 헤세,민음사,20020120.0,소설
4,5,9791124137635,S000220693844,한국사 이상현상 연구원(일반판),"22,000","19,800",10%,5%,"1,100",최인서,다이브,20260831.0,소설


### 실행 후 확인할 것

원본 파일은 약 1,000개의 베스트셀러 상품을 포함하고 있다.
실제 행과 컬럼 개수는 **본인 Notebook의 출력값**으로 직접 확인한다.

---
# 실습 3. 분석에 필요한 컬럼만 선택하기

## 해야 할 일

원본의 모든 컬럼이 이후 분석에 필요한 것은 아니다.
사용하지 않는 데이터를 계속 들고 있으면 분석 과정이 복잡해진다.

이번 과정에서 사용할 컬럼은 다음 8개다.

| 컬럼 | 쓰임 |
|---|---|
| 순위 | 베스트셀러 순위 |
| 판매상품 ID | 개별 상품을 구분하는 식별자 |
| 상품명 | **텍스트 분석의 핵심 입력 데이터** |
| 판매가 | 가격 분석 |
| 인물 | 저자 |
| 출판사 | 출판사별 분석 |
| 발행(출시)일자 | 시기별 분석 |
| 분야 | **카테고리 분류의 정답 데이터** |

## 프롬프트 예시

```
pandas DataFrame df에 교보문고 베스트셀러 데이터가 있습니다.
다음 컬럼만 선택해서 새로운 DataFrame df_books를 만들고 싶습니다.
- 순위
- 판매상품 ID
- 상품명
- 판매가
- 인물
- 출판사
- 발행(출시)일자
- 분야
Python 초보자가 이해할 수 있도록 가장 간단한 pandas 코드로 작성해 주세요.
컬럼이 실제로 잘 선택되었는지 확인하는 코드도 함께 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [6]:
# 사용할 컬럼 목록
use_cols = ["순위", "판매상품 ID", "상품명", "판매가", "인물", "출판사", "발행(출시)일자", "분야"]

df_books = df[use_cols].copy()   # copy()로 원본과 분리

print("선택 후 크기:", df_books.shape)
print("선택된 컬럼:", df_books.columns.tolist())
df_books.head()

선택 후 크기: (986, 8)
선택된 컬럼: ['순위', '판매상품 ID', '상품명', '판매가', '인물', '출판사', '발행(출시)일자', '분야']


,순위,판매상품 ID,상품명,판매가,인물,출판사,발행(출시)일자,분야
0,1,S000220308313,"세네카, 오늘을 빼앗기고 있는 당신에게","16,200",세네카,논픽션,20260701.0,인문
1,2,S000221248222,흔한남매 23,"15,120",흔한남매,미래엔아이세움,20260916.0,어린이(초등)
2,3,S000221075112,머니 트렌드 2027,"24,750",정태익 외,북모먼트,20260916.0,경제/경영
3,4,S000000620195,싯다르타,"7,200",헤르만 헤세,민음사,20020120.0,소설
4,5,S000220693844,한국사 이상현상 연구원(일반판),"19,800",최인서,다이브,20260831.0,소설


### 실행 후 확인할 것

`df_books.columns`에 원하는 8개 컬럼만 남았는지 직접 확인한다.

---
# 실습 4. 컬럼 이름을 사용하기 쉽게 정리하기

## 해야 할 일

원본 컬럼 이름은 사람이 읽기에는 좋지만 코드에서 반복 사용하기에는 길다.

| 원본 | 변경 후 |
|---|---|
| 판매상품 ID | 판매상품ID |
| 인물 | 저자 |
| 발행(출시)일자 | 발행일 |

나머지 컬럼은 그대로 사용한다.

## 프롬프트 예시

```
DataFrame df_books의 컬럼 이름을 일부 변경하려고 합니다.
판매상품 ID -> 판매상품ID
인물 -> 저자
발행(출시)일자 -> 발행일
pandas rename()을 이용해 가장 간단하게 변경하는 코드를 작성해 주세요.
변경 후 columns를 출력해서 결과를 확인하는 코드도 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [7]:
df_books = df_books.rename(columns={
    "판매상품 ID": "판매상품ID",
    "인물": "저자",
    "발행(출시)일자": "발행일"
})

print(df_books.columns.tolist())
df_books.head()

['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']


,순위,판매상품ID,상품명,판매가,저자,출판사,발행일,분야
0,1,S000220308313,"세네카, 오늘을 빼앗기고 있는 당신에게","16,200",세네카,논픽션,20260701.0,인문
1,2,S000221248222,흔한남매 23,"15,120",흔한남매,미래엔아이세움,20260916.0,어린이(초등)
2,3,S000221075112,머니 트렌드 2027,"24,750",정태익 외,북모먼트,20260916.0,경제/경영
3,4,S000000620195,싯다르타,"7,200",헤르만 헤세,민음사,20020120.0,소설
4,5,S000220693844,한국사 이상현상 연구원(일반판),"19,800",최인서,다이브,20260831.0,소설


---
# 실습 5. 결측치 확인하기

## 해야 할 일

결측치는 값이 비어 있는 데이터다.
데이터 분석에서는 결측치를 **무조건 삭제하지 않는다.**
먼저 어디에 얼마나 있는지 확인하고, 분석 목적에 맞게 처리 방법을 결정한다.

## 프롬프트 예시

```
DataFrame df_books의 컬럼별 결측치 개수를 확인하고 싶습니다.
pandas isna()와 sum()을 이용해 가장 간단한 코드를 작성해 주세요.
그리고 결측치가 있는 컬럼만 보고 싶을 때 사용할 수 있는 쉬운 코드도 알려주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [8]:
# 컬럼별 결측치 개수
print("컬럼별 결측치 개수:")
print(df_books.isna().sum())
print()

# 결측치가 1개 이상인 컬럼만 보기
na_count = df_books.isna().sum()
print("결측치가 있는 컬럼만:")
print(na_count[na_count > 0])

컬럼별 결측치 개수:
순위         0
판매상품ID     0
상품명        0
판매가        0
저자        10
출판사       11
발행일        8
분야        64
dtype: int64

결측치가 있는 컬럼만:
저자     10
출판사    11
발행일     8
분야     64
dtype: int64


## 여기서 바로 dropna()를 실행하지 않는다

분야가 없는 책은 이후 **카테고리 분류 학습**에는 쓸 수 없을지 몰라도,
**제목 단어 빈도 분석**에는 사용할 수 있다.

즉 결측치 처리 방법은 분석 목적에 따라 달라진다.

### 이번 Chapter의 처리 기준

- 저자 결측치 -> `"미상"`
- 분야 결측치 -> `"미분류"`

## 프롬프트 예시

```
DataFrame df_books에서
저자 컬럼의 결측치는 '미상',
분야 컬럼의 결측치는 '미분류'로 바꾸고 싶습니다.
pandas fillna()를 이용해서 초보자가 이해하기 쉬운 코드로 작성해 주세요.
처리 후 결측치 개수를 다시 확인하는 코드도 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [9]:
df_books["저자"] = df_books["저자"].fillna("미상")
df_books["분야"] = df_books["분야"].fillna("미분류")

# 처리 후 반드시 다시 확인
print("처리 후 결측치 개수:")
print(df_books.isna().sum())

처리 후 결측치 개수:
순위         0
판매상품ID     0
상품명        0
판매가        0
저자         0
출판사       11
발행일        8
분야         0
dtype: int64


### 기억할 것

전처리는 코드를 실행하는 것으로 끝나지 않는다. **처리 결과를 다시 확인해야 한다.**

---
# 실습 6. 판매가를 숫자로 바꾸기

## 해야 할 일

Excel에서 판매가가 다음처럼 저장되어 있을 수 있다.

```
16,200
17,820
7,200
```

사람이 보기엔 숫자지만 pandas에서는 **문자열**로 읽힌다.
문자열 상태에서는 평균, 합계 같은 계산을 제대로 할 수 없다.
쉼표를 제거하고 숫자 자료형으로 변환한다.

## 프롬프트 예시

```
DataFrame df_books의 판매가 컬럼에
'16,200', '17,820'처럼 쉼표가 포함된 가격 문자열이 있습니다.
1. 쉼표를 제거하고
2. 숫자형으로 변환하고
3. 변환 후 dtype을 확인하고 싶습니다.
pandas를 사용해 초보자가 이해하기 쉬운 코드로 작성해 주세요.
가능하면 한 단계씩 설명해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [10]:
# 변환 전 자료형 확인
print("변환 전 dtype:", df_books["판매가"].dtype)
print(df_books["판매가"].head())

변환 전 dtype: str
0    16,200
1    15,120
2    24,750
3     7,200
4    19,800
Name: 판매가, dtype: str


## 프롬프트 예시

```
DataFrame df_books의 판매가 컬럼에
'16,200', '17,820'처럼 쉼표가 포함된 가격 문자열이 있습니다.
쉼표를 제거하고 숫자형으로 변환한 뒤 dtype을 확인하고 싶습니다.
숫자로 바꿀 수 없는 값은 결측 처리해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [11]:
# 1) 문자열로 바꾼 뒤 쉼표 제거
# 2) 숫자형으로 변환 (숫자로 바꿀 수 없는 값은 NaN 처리)
df_books["판매가"] = df_books["판매가"].astype(str).str.replace(",", "")
df_books["판매가"] = pd.to_numeric(df_books["판매가"], errors="coerce")

print("변환 후 dtype:", df_books["판매가"].dtype)
print(df_books["판매가"].head())

변환 후 dtype: int64
0    16200
1    15120
2    24750
3     7200
4    19800
Name: 판매가, dtype: int64


## 프롬프트 예시

```
판매가 컬럼을 숫자형으로 바꿨습니다.
이상하게 변환된 값이 없는지 확인하고 싶습니다.
describe()로 기본 통계를 보고, 변환에 실패해 결측이 된 개수도 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [12]:
# 간단한 통계로 이상한 값이 없는지 확인
print(df_books["판매가"].describe())
print()
print("변환 실패(결측)한 개수:", df_books["판매가"].isna().sum())

count       986.000000
mean      17581.967546
std        9321.078965
min        2690.000000
25%       13500.000000
50%       16200.000000
75%       19800.000000
max      129600.000000
Name: 판매가, dtype: float64

변환 실패(결측)한 개수: 0


### 실행 후 확인할 것

- dtype이 숫자형(int64 또는 float64)으로 바뀌었는가
- describe() 결과에서 지나치게 크거나 작게 변환된 값은 없는가
- 변환 실패 개수가 0인가 (0이 아니면 원본에 어떤 값이 있었는지 확인)

---
# 실습 7. 발행일을 날짜형으로 바꾸기

## 해야 할 일

원본의 발행일은 다음처럼 보일 수 있다.

```
20260701
20260831
20020120
```

연도-월-일 정보를 가지고 있지만 바로 날짜 계산에 쓰기 어렵다.
날짜 자료형으로 변환하면 연도, 월, 날짜 단위 분석이 쉬워진다.

## 프롬프트 예시

```
DataFrame df_books의 발행일 컬럼이
20260701처럼 YYYYMMDD 형식으로 저장되어 있습니다.
pandas to_datetime()을 이용해 날짜 자료형으로 변환하고 싶습니다.
format을 명시해서 변환하는 가장 쉬운 코드를 작성해 주세요.
변환 후 dtype과 앞의 5개 값을 확인하는 코드도 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [13]:
print("변환 전 dtype:", df_books["발행일"].dtype)
print(df_books["발행일"].head())

변환 전 dtype: float64
0    20260701.0
1    20260916.0
2    20260916.0
3    20020120.0
4    20260831.0
Name: 발행일, dtype: float64


### 주의: 발행일이 float64로 읽힌 경우

위 셀의 dtype이 `float64`이고 값이 `20260701.0`처럼 보인다면,
발행일에 **결측치가 섞여 있어서** pandas가 정수 대신 실수로 읽은 것이다.

이 상태에서 그냥 `astype(str)`을 하면 `"20260701.0"`이 되어
`format="%Y%m%d"` 변환이 **전부 실패**한다.

그래서 문자열로 바꾸기 전에 **정수로 먼저 바꾼다.**
결측치가 있어도 쓸 수 있도록 대문자 `Int64`를 사용한다.

## 프롬프트 예시

```
DataFrame df_books의 발행일 컬럼이
20260701.0처럼 소수점이 붙은 실수로 읽혔습니다.
결측치가 섞여 있어서 정수가 아니라 실수가 된 상태입니다.
먼저 정수로 바꾼 뒤 YYYYMMDD 형식의 날짜 자료형으로 변환하고 싶습니다.
변환 후 dtype과 앞의 5개 값, 변환 실패 개수를 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [14]:
# 1) 정수로 변환 (결측치가 있어도 되도록 대문자 Int64 사용)
# 2) 문자열로 변환
# 3) YYYYMMDD 형식으로 날짜 변환 (변환 안 되는 값은 NaT 처리)
df_books["발행일"] = pd.to_datetime(
    df_books["발행일"].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce"
)

print("변환 후 dtype:", df_books["발행일"].dtype)
print(df_books["발행일"].head())
print()
print("변환 실패(결측)한 개수:", df_books["발행일"].isna().sum())

변환 후 dtype: datetime64[us]
0   2026-07-01
1   2026-09-16
2   2026-09-16
3   2002-01-20
4   2026-08-31
Name: 발행일, dtype: datetime64[us]

변환 실패(결측)한 개수: 8


### 실행 후 확인할 것

정상 변환되면 다음과 비슷하게 표시된다.

```
2026-07-01
2026-08-31
2002-01-20
```

변환 실패 개수는 **원본 발행일 결측치 개수와 같아야 한다.**
그보다 많다면 형식이 다른 값이 섞여 있다는 뜻이므로 원본을 다시 확인한다.

---
# 실습 8. 문자열 앞뒤 공백 정리하기

## 해야 할 일

상품명, 저자, 출판사, 분야는 문자열 데이터다.
앞뒤에 불필요한 공백이 있으면 같은 값도 서로 다른 데이터처럼 처리된다.

```
"소설"
"소설 "
```

사람이 보기엔 같지만 문자열로는 다르다.

## 프롬프트 예시

```
DataFrame df_books에서 다음 문자열 컬럼의 앞뒤 공백을 제거하고 싶습니다.
상품명
저자
출판사
분야
pandas 문자열 함수 str.strip()을 이용해
초보자가 이해하기 쉬운 반복문 코드로 작성해 주세요.
처리 후 각 컬럼의 앞의 몇 개 값을 확인하는 방법도 알려주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [15]:
text_cols = ["상품명", "저자", "출판사", "분야"]

for col in text_cols:
    df_books[col] = df_books[col].astype(str).str.strip()

# 처리 결과 확인
for col in text_cols:
    print("[" + col + "]")
    print(df_books[col].head(3).tolist())
    print()

[상품명]
['세네카, 오늘을 빼앗기고 있는 당신에게', '흔한남매 23', '머니 트렌드 2027']

[저자]
['세네카', '흔한남매', '정태익 외']

[출판사]
['논픽션', '미래엔아이세움', '북모먼트']

[분야]
['인문', '어린이(초등)', '경제/경영']



이번 Chapter에서는 **앞뒤 공백 정도만** 정리한다.

특수문자 제거, 단어 분리, 품사 분석, 불용어 제거 같은 본격적인 텍스트 전처리는 Chapter 02에서 진행한다.

---
# 실습 9. 중복 데이터 확인하기

## 해야 할 일

중복을 발견했다고 바로 삭제하면 안 된다.
먼저 **무엇을 기준으로 중복인지** 생각해야 한다.

같은 상품명이 여러 번 등장할 수 있지만,
같은 제목이라도 다른 판본, 다른 상품, 개정판일 수 있다.

따라서 상품명만 보고 중복 행을 삭제하지 않는다.
우선 상품을 구분하는 판매상품ID를 확인한다.

## 프롬프트 예시

```
DataFrame df_books에서 중복 데이터를 확인하고 싶습니다.
1. 전체 행이 완전히 같은 중복 행 개수
2. 판매상품ID가 중복된 행 개수
3. 상품명이 중복된 행 개수
을 각각 pandas duplicated()로 확인하는 간단한 코드를 작성해 주세요.
중복을 바로 삭제하지 말고 개수만 확인하도록 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [16]:
print("전체 행이 완전히 같은 중복 행 개수:", df_books.duplicated().sum())
print("판매상품ID가 중복된 행 개수:", df_books.duplicated(subset=["판매상품ID"]).sum())
print("상품명이 중복된 행 개수:", df_books.duplicated(subset=["상품명"]).sum())

전체 행이 완전히 같은 중복 행 개수: 0
판매상품ID가 중복된 행 개수: 0
상품명이 중복된 행 개수: 45


## 프롬프트 예시

```
DataFrame df_books에서 상품명이 중복된 행이 실제로 어떤 데이터인지
눈으로 확인하고 싶습니다. 삭제는 하지 마세요.
중복에 해당하는 행 개수를 출력하고,
상품명 순으로 정렬해 앞의 10개를 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [17]:
# 상품명이 중복된 행이 실제로 어떤 데이터인지 눈으로 확인만 한다 (삭제하지 않음)
dup_titles = df_books[df_books.duplicated(subset=["상품명"], keep=False)]
print("중복 상품명에 해당하는 행 개수:", len(dup_titles))
dup_titles.sort_values("상품명").head(10)

중복 상품명에 해당하는 행 개수: 88


,순위,판매상품ID,상품명,판매가,저자,출판사,발행일,분야
714,715,S000221061899,2026 하반기 해커스 GSAT 삼성직무적성검사 실전모의고사 12회분,19710,해커스 GSAT 취업교육연구소,해커스잡,2026-09-01,취업/수험서
682,683,E000013559679,2026 하반기 해커스 GSAT 삼성직무적성검사 실전모의고사 12회분,21900,해커스 GSAT 취업교육연구소,해커스잡,2026-09-08,미분류
102,103,S000218914182,2026 해커스 GSAT 삼성직무적성검사 통합 기본서 최신기출유형+실전모의고사 (수...,21600,해커스 취업교육연구소,해커스잡,2026-01-02,취업/수험서
138,139,E000012435877,2026 해커스 GSAT 삼성직무적성검사 통합 기본서 최신기출유형+실전모의고사 (수...,22600,해커스 취업교육연구소,해커스잡,2026-01-09,미분류
377,378,E000012809329,"AI, 신의 탄생 인간의 종말",15400,엘리에저 유드코스키 외,상상스퀘어,2026-04-03,미분류
99,100,S000219568490,"AI, 신의 탄생 인간의 종말",19800,엘리에저 유드코스키 외,상상스퀘어,2026-04-08,경제/경영
103,104,S000221281513,가브리엘의 왕초보 영어회화,17010,Thomas Gabrielle Allanta,상상스퀘어,2026-09-16,외국어
241,242,E000013582364,가브리엘의 왕초보 영어회화,13230,Thomas Gabrielle Allanta,상상스퀘어,2026-09-15,미분류
70,71,S000200550189,급류,12600,정대건,민음사,2022-12-22,소설
830,831,E000005162329,급류,9800,정대건,민음사,2023-03-21,미분류


### 중요한 생각

> 상품명이 같다  ≠  반드시 잘못된 중복 데이터다

중복 데이터는 먼저 의미를 확인한 뒤 처리한다.
이번 단계에서는 중복 여부만 확인하고 원본 행을 임의로 삭제하지 않는다.

---
# 실습 10. 전처리 결과 검증하기

## 해야 할 일

지금까지 여러 전처리를 수행했다.
마지막에는 반드시 데이터가 의도한 형태가 되었는지 다시 확인한다.

## 프롬프트 예시

```
전처리가 끝난 pandas DataFrame df_books를 최종 검증하고 싶습니다.
다음 내용을 한 번에 확인하는 초보자용 코드를 작성해 주세요.
- shape
- columns
- dtypes
- isna().sum()
- 판매가 앞의 5개 값
- 발행일 앞의 5개 값
- 판매상품ID 중복 개수
- head()
결과를 자동으로 좋다/나쁘다 판단하지 말고,
확인할 수 있도록 print 중심으로 간단하게 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [18]:
print("=== 최종 검증 ===")
print()

print("1. shape:", df_books.shape)
print()

print("2. columns:")
print(df_books.columns.tolist())
print()

print("3. dtypes:")
print(df_books.dtypes)
print()

print("4. 결측치 개수:")
print(df_books.isna().sum())
print()

print("5. 판매가 앞의 5개 값:")
print(df_books["판매가"].head().tolist())
print()

print("6. 발행일 앞의 5개 값:")
print(df_books["발행일"].head().tolist())
print()

print("7. 판매상품ID 중복 개수:", df_books.duplicated(subset=["판매상품ID"]).sum())

=== 최종 검증 ===

1. shape: (986, 8)

2. columns:
['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']

3. dtypes:
순위                 int64
판매상품ID               str
상품명                  str
판매가                int64
저자                   str
출판사                  str
발행일       datetime64[us]
분야                   str
dtype: object

4. 결측치 개수:
순위         0
판매상품ID     0
상품명        0
판매가        0
저자         0
출판사       11
발행일        8
분야         0
dtype: int64

5. 판매가 앞의 5개 값:
[16200, 15120, 24750, 7200, 19800]

6. 발행일 앞의 5개 값:
[Timestamp('2026-07-01 00:00:00'), Timestamp('2026-09-16 00:00:00'), Timestamp('2026-09-16 00:00:00'), Timestamp('2002-01-20 00:00:00'), Timestamp('2026-08-31 00:00:00')]

7. 판매상품ID 중복 개수: 0


## 프롬프트 예시

```
전처리가 끝난 DataFrame df_books의 앞부분을
표 형태로 확인하고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [19]:
df_books.head()

,순위,판매상품ID,상품명,판매가,저자,출판사,발행일,분야
0,1,S000220308313,"세네카, 오늘을 빼앗기고 있는 당신에게",16200,세네카,논픽션,2026-07-01,인문
1,2,S000221248222,흔한남매 23,15120,흔한남매,미래엔아이세움,2026-09-16,어린이(초등)
2,3,S000221075112,머니 트렌드 2027,24750,정태익 외,북모먼트,2026-09-16,경제/경영
3,4,S000000620195,싯다르타,7200,헤르만 헤세,민음사,2002-01-20,소설
4,5,S000220693844,한국사 이상현상 연구원(일반판),19800,최인서,다이브,2026-08-31,소설


코드가 오류 없이 실행되는 것만 확인하지 않는다.
**출력된 값이 내가 기대한 데이터인지 직접 읽어본다.**

---
# 실습 11. 전처리 데이터 저장하기

## 해야 할 일

다음 Chapter에서 이 데이터를 다시 사용한다.
전처리한 DataFrame을 CSV 파일로 저장한다.

- 파일명: `book_bestseller_clean.csv`
- 한글이 Excel에서 깨지지 않도록 `utf-8-sig` 인코딩 사용
- DataFrame 인덱스는 저장하지 않음

## 프롬프트 예시

```
전처리가 끝난 pandas DataFrame df_books를
book_bestseller_clean.csv 파일로 저장하고 싶습니다.
Windows와 Excel에서도 한글이 잘 보이도록 utf-8-sig 인코딩을 사용하고,
DataFrame의 인덱스는 저장하지 않도록 해주세요.
pandas to_csv()를 이용해 한 줄의 간단한 코드로 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [20]:
df_books.to_csv("book_bestseller_clean.csv", index=False, encoding="utf-8-sig")
print("저장 완료: book_bestseller_clean.csv")

저장 완료: book_bestseller_clean.csv


## 프롬프트 예시

```
방금 저장한 book_bestseller_clean.csv가 제대로 만들어졌는지
다시 읽어서 확인하고 싶습니다.
크기와 앞부분을 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [21]:
# 저장한 파일이 제대로 만들어졌는지 다시 읽어서 확인
check = pd.read_csv("book_bestseller_clean.csv")
print(check.shape)
check.head()

(986, 8)


,순위,판매상품ID,상품명,판매가,저자,출판사,발행일,분야
0,1,S000220308313,"세네카, 오늘을 빼앗기고 있는 당신에게",16200,세네카,논픽션,2026-07-01,인문
1,2,S000221248222,흔한남매 23,15120,흔한남매,미래엔아이세움,2026-09-16,어린이(초등)
2,3,S000221075112,머니 트렌드 2027,24750,정태익 외,북모먼트,2026-09-16,경제/경영
3,4,S000000620195,싯다르타,7200,헤르만 헤세,민음사,2002-01-20,소설
4,5,S000220693844,한국사 이상현상 연구원(일반판),19800,최인서,다이브,2026-08-31,소설


---
# 보충. 온라인 주간 베스트셀러도 같은 방식으로 전처리하기

## 해야 할 일

교보문고에서 **온라인 일간 베스트**와 **온라인 주간 베스트**를 각각 내려받았다.
위에서 일간 데이터로 전처리 흐름을 모두 익혔으니,
주간 데이터에는 **똑같은 순서를 한 번에 적용**해 본다.

같은 작업을 반복할 때는 사람이 코드를 매번 다시 쓰지 않고
**앞에서 검증한 순서를 그대로 재사용**한다는 점을 확인하는 단계다.

> 주의: 처리 결과는 여기서도 반드시 다시 확인한다. 일간과 주간은 결측치 개수가 다르다.

## 프롬프트 예시

```
교보문고_온라인_주간_베스트셀러_상품리스트.xlsx 파일이
Notebook과 같은 폴더에 있습니다.
df_week_raw라는 DataFrame으로 불러오고
크기와 컬럼별 결측치 개수를 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [22]:
# 주간 파일 불러오기
df_week_raw = pd.read_excel("교보문고_온라인_주간_베스트셀러_상품리스트.xlsx")
print("주간 원본 크기:", df_week_raw.shape)
print("주간 원본 결측치:")
print(df_week_raw.isna().sum())

주간 원본 크기: (983, 13)
주간 원본 결측치:


순위           0
상품코드         1
판매상품 ID      0
상품명          0
정가           0
판매가          0
할인율          0
적립율          0
적립예정포인트      0
인물           4
출판사          5
발행(출시)일자     3
분야          49
dtype: int64

C:\Users\apll1\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


## 프롬프트 예시

```
일간 데이터에 했던 전처리를 주간 데이터 df_week_raw에도
똑같이 적용하고 싶습니다. 순서는 다음과 같습니다.
1. use_cols에 담긴 컬럼만 선택
2. 판매상품 ID/인물/발행(출시)일자 컬럼명 정리
3. 저자는 '미상', 분야는 '미분류'로 결측치 채우기
4. 판매가에서 쉼표를 빼고 숫자형으로 변환
5. 발행일을 YYYYMMDD 날짜형으로 변환
6. text_cols의 앞뒤 공백 제거
단계마다 번호 주석을 달고, 마지막에 크기·자료형·결측치를 출력해 주세요.
try/except 예외 처리는 넣지 마세요.
```

In [23]:
# 일간에 적용했던 순서를 그대로 적용

# 1) 필요한 컬럼 선택
df_week = df_week_raw[use_cols].copy()

# 2) 컬럼 이름 정리
df_week = df_week.rename(columns={
    "판매상품 ID": "판매상품ID",
    "인물": "저자",
    "발행(출시)일자": "발행일"
})

# 3) 결측치 채우기
df_week["저자"] = df_week["저자"].fillna("미상")
df_week["분야"] = df_week["분야"].fillna("미분류")

# 4) 판매가 숫자형 변환
df_week["판매가"] = df_week["판매가"].astype(str).str.replace(",", "")
df_week["판매가"] = pd.to_numeric(df_week["판매가"], errors="coerce")

# 5) 발행일 날짜형 변환
df_week["발행일"] = pd.to_datetime(
    df_week["발행일"].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="coerce"
)

# 6) 문자열 앞뒤 공백 제거
for col in text_cols:
    df_week[col] = df_week[col].astype(str).str.strip()

print("전처리 후 크기:", df_week.shape)
print()
print("dtypes:")
print(df_week.dtypes)
print()
print("결측치 개수:")
print(df_week.isna().sum())

전처리 후 크기: (983, 8)

dtypes:
순위                 int64
판매상품ID               str
상품명                  str
판매가                int64
저자                   str
출판사                  str
발행일       datetime64[us]
분야                   str
dtype: object

결측치 개수:


순위        0
판매상품ID    0
상품명       0
판매가       0
저자        0
출판사       5
발행일       3
분야        0
dtype: int64


## 프롬프트 예시

```
전처리가 끝난 일간 데이터 df_books와 주간 데이터 df_week가 있습니다.
두 데이터를 간단히 비교하고 싶습니다.
행 수, 판매가 평균, 순위 1위 도서명을 나란히 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [24]:
# 일간과 주간을 나란히 비교
print("일간 행 수:", len(df_books), " / 주간 행 수:", len(df_week))
print()
print("일간 판매가 평균:", round(df_books["판매가"].mean()))
print("주간 판매가 평균:", round(df_week["판매가"].mean()))
print()
print("일간 1위:", df_books.loc[df_books["순위"] == 1, "상품명"].tolist())
print("주간 1위:", df_week.loc[df_week["순위"] == 1, "상품명"].tolist())

일간 행 수: 986  / 주간 행 수: 983

일간 판매가 평균: 17582
주간 판매가 평균: 18485

일간 1위: ['세네카, 오늘을 빼앗기고 있는 당신에게']
주간 1위: ['세네카, 오늘을 빼앗기고 있는 당신에게']


## 프롬프트 예시

```
전처리가 끝난 주간 데이터 df_week를
book_bestseller_weekly_clean.csv로 저장하고 싶습니다.
utf-8-sig 인코딩을 쓰고 인덱스는 저장하지 말아 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [25]:
# 주간 데이터도 CSV로 저장
df_week.to_csv("book_bestseller_weekly_clean.csv", index=False, encoding="utf-8-sig")
print("저장 완료: book_bestseller_weekly_clean.csv")

저장 완료: book_bestseller_weekly_clean.csv


---
# 보충 2. 전처리한 데이터 살펴보기

## 해야 할 일

전처리는 **다음 분석을 위한 준비**다.
준비가 끝났으면 데이터가 실제로 어떤 모습인지 한 번 훑어본다.

이 단계에서 확인한 내용은 Chapter 02(단어 빈도)와 Chapter 03(벡터화)의
결과를 해석할 때 근거가 된다.

- 어떤 분야의 책이 많은가
- 가격은 어느 구간에 몰려 있는가
- 언제 나온 책들인가
- 어떤 출판사·저자가 많은가
- 이상하게 보이는 데이터는 없는가

## 프롬프트 예시

```
전처리가 끝난 DataFrame df_books가 있습니다.
어떤 분야의 책이 많은지 보고 싶습니다.
분야 종류가 몇 개인지, 도서 수 상위 10개 분야,
그리고 '미분류'가 전체의 몇 퍼센트인지 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [26]:
# 1) 분야 분포
print("분야 종류 수:", df_books["분야"].nunique())
print()
print("분야별 도서 수 상위 10개:")
print(df_books["분야"].value_counts().head(10))
print()
print("미분류 비중:", round((df_books["분야"] == "미분류").mean() * 100, 1), "%")

분야 종류 수:

 38

분야별 도서 수 상위 10개:
분야
소설         178
인문         100
어린이(초등)     84
취업/수험서      79
시/에세이       73
경제/경영       69
미분류         64
외국어         58
자기계발        57
만화          41
Name: count, dtype: int64

미분류 비중: 6.5 %


## 프롬프트 예시

```
DataFrame df_books의 판매가 컬럼이 숫자형입니다.
평균, 중앙값, 최소, 최대 같은 기본 통계를 한 번에 보고 싶습니다.
소수점은 없애 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [27]:
# 2) 판매가 분포
print(df_books["판매가"].describe().round(0))

count       986.0
mean      17582.0
std        9321.0
min        2690.0
25%       13500.0
50%       16200.0
75%       19800.0
max      129600.0
Name: 판매가, dtype: float64


## 프롬프트 예시

```
DataFrame df_books의 판매가를 구간으로 나눠서
어느 가격대에 책이 몰려 있는지 보고 싶습니다.
구간은 1만 미만, 1만~1.5만, 1.5만~2만, 2만~3만, 3만~5만, 5만 이상으로 하고
구간 순서대로 도서 수를 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [28]:
# 3) 가격 구간별 도서 수
bins = [0, 10000, 15000, 20000, 30000, 50000, 10**9]
labels = ["1만 미만", "1만~1.5만", "1.5만~2만", "2만~3만", "3만~5만", "5만 이상"]

price_band = pd.cut(df_books["판매가"], bins=bins, labels=labels, right=False)
print(price_band.value_counts().sort_index())

판매가
1만 미만      111
1만~1.5만    212
1.5만~2만    445
2만~3만      168
3만~5만       39
5만 이상       11
Name: count, dtype: int64


## 프롬프트 예시

```
DataFrame df_books의 발행일이 날짜형입니다.
연도별로 몇 권인지 최근 8년만 보고 싶습니다.
그리고 2026년 이후에 나온 책이 전체의 몇 퍼센트인지도 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [29]:
# 4) 발행 연도 분포
year = df_books["발행일"].dt.year

print("연도별 도서 수 (최근 8년):")
print(year.value_counts().sort_index(ascending=False).head(8))
print()
print("2026년 이후 발행 비중:", round((year >= 2026).mean() * 100, 1), "%")

연도별 도서 수 (최근 8년):
발행일
2026.0    541
2025.0    121
2024.0     65
2023.0     46
2022.0     32
2021.0     32
2020.0     13
2019.0     19
Name: count, dtype: int64

2026년 이후 발행 비중: 54.9 %


## 프롬프트 예시

```
DataFrame df_books에서 출판사와 저자가 각각
많이 등장한 상위 10개를 보고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [30]:
# 5) 출판사와 저자 상위 10개
print("출판사 상위 10개:")
print(df_books["출판사"].value_counts().head(10))
print()
print("저자 상위 10개:")
print(df_books["저자"].value_counts().head(10))

출판사 상위 10개:
출판사
민음사         65
교보문고        21
문학동네        21
대원씨아이       19
미래엔아이세움     18
상상스퀘어       16
해커스어학연구소    16
시원스쿨닷컴      14
김영사         12
창비          12
Name: count, dtype: int64

저자 상위 10개:
저자
교보문고        16
미상          10
호메로스         9
헤르만 헤세       8
ETS          8
흔한남매         7
최태성          7
심우철          7
히가시노 게이고     6
설민석 외        6
Name: count, dtype: int64


## 프롬프트 예시

```
DataFrame df_books에서 판매가가 가장 비싼 5개와
가장 싼 5개를 보고 싶습니다.
상품명, 판매가, 분야 세 컬럼만 표로 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [31]:
# 6) 가장 비싼 책과 가장 싼 책
print("가장 비싼 5개:")
print(df_books.nlargest(5, "판매가")[["상품명", "판매가", "분야"]].to_string(index=False))
print()
print("가장 싼 5개:")
print(df_books.nsmallest(5, "판매가")[["상품명", "판매가", "분야"]].to_string(index=False))

가장 비싼 5개:
                            상품명    판매가         분야
데뷔 못 하면 죽는 병 걸림 5부 굿즈박스 세트(한정판) 129600         소설
                교보문고 기프트카드 10만원 100000 상품권(기프트카드)
    2027 해커스경찰 갓대환 형사법 기출총정리 세트  89100     취업/수험서
             2027 Trend Preview  70000         강연
  The Scent of Page : 디퓨저 300ML  69700         책향

가장 싼 5개:
                              상품명  판매가 분야
  싯다르타(초판본)(1922년 오리지널 초판본 표지디자인) 2690 소설
   이방인(초판본)(1944년 오리지널 초판본 표지디자인) 2690 소설
  데미안(초판본)(1919년 오리지널 초판본 표지 디자인) 2690 소설
노인과 바다(초판본)(1952년 오리지널 초판본 표지디자인) 2690 소설
 어린 왕자(초판본)(1943년 오리지널 초판본 표지디자인) 2700 소설


## 프롬프트 예시

```
DataFrame df_books에서 분야별 평균 판매가를 보고 싶습니다.
도서가 20권 이상인 분야만 남기고, 평균이 싼 순서로 정렬해 주세요.
분야별 도서 수도 함께 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [32]:
# 7) 분야별 평균 판매가 (20권 이상인 분야만)
price_by_field = (
    df_books.groupby("분야")["판매가"]
    .agg(["count", "mean"])
    .query("count >= 20")
    .sort_values("mean")
    .round(0)
)
print(price_by_field)

         count     mean
분야                     
만화          41   8767.0
어린이(초등)     84  14048.0
소설         178  14798.0
청소년         24  14812.0
시/에세이       73  15418.0
미분류         64  16315.0
자기계발        57  17566.0
외국어         58  18098.0
인문         100  18911.0
경제/경영       69  21402.0
취업/수험서      79  28114.0


## 여기서 발견한 것 — 도서가 아닌 상품이 섞여 있다

가장 비싼 상품 목록을 보면 다음이 나온다.

```
교보문고 기프트카드 10만원              분야: 상품권(기프트카드)
The Scent of Page : 디퓨저 300ML       분야: 책향
데뷔 못 하면 죽는 병 걸림 5부 굿즈박스 세트  분야: 소설
```

**상품권, 디퓨저, 굿즈가 "베스트셀러 상품 리스트"에 함께 들어 있다.**
저자 상위 목록에 `교보문고`가 1위로 올라온 것도 같은 이유다.
이 상품들의 저자 칸에 출판사 대신 `교보문고`가 들어가 있기 때문이다.

이건 데이터가 잘못된 것이 아니라,
**우리가 받은 파일이 "도서 목록"이 아니라 "상품 목록"이기 때문**이다.

### 왜 이걸 미리 알아야 할까?

Chapter 02에서 제목 단어를 셀 때 `디퓨저`, `기프트카드` 같은 단어가 섞여 들어온다.
Chapter 04에서 분야를 분류할 때도 `상품권`, `책향` 같은 분야가 정답 데이터에 들어간다.

지금 단계에서는 **삭제하지 않는다.**
몇 건이 그런 상품인지 먼저 확인하고, 분석 목적이 정해졌을 때 판단한다.

## 프롬프트 예시

```
DataFrame df_books에 상품권, 책향, 굿즈, 강연처럼
도서가 아닌 상품이 섞여 있는 것을 발견했습니다.
이 분야에 해당하는 상품이 몇 건인지 세고,
상품명·분야·판매가를 10개만 확인하고 싶습니다.
삭제는 하지 마세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [33]:
# 8) 도서가 아닌 것으로 보이는 분야가 몇 건인지 확인 (삭제하지 않음)
non_book_fields = ["상품권(기프트카드)", "책향", "교보문고 굿즈", "강연"]

mask = df_books["분야"].isin(non_book_fields)
print("도서가 아닌 것으로 보이는 상품 수:", mask.sum())
print()
print(df_books.loc[mask, ["상품명", "분야", "판매가"]].head(10).to_string(index=False))

도서가 아닌 것으로 보이는 상품 수: 20

                              상품명         분야    판매가
The Scent of Page : 디퓨저 리필액 250ML         책향  23800
    The Scent of Page : 디퓨저 200ML         책향  49300
 The Scent of Page : 차량용 방향제(개선판)         책향  35700
   The Scent of Page : 룸스프레이 80ML         책향  25500
                        교보문고 리딩백S    교보문고 굿즈   2900
                        교보문고 리딩백M    교보문고 굿즈   3900
               2027 Trend Preview         강연  70000
           The Scent of Page : 샤쉐         책향  17000
                  교보문고 기프트카드 10만원 상품권(기프트카드) 100000
    The Scent of Page : 디퓨저 300ML         책향  69700


---
# 12. 전처리 결과 정리 (Markdown)

## 프롬프트 예시

인공지능이 실행 결과를 **추측하게 하면 안 된다.**
Notebook에서 실제로 확인한 숫자를 프롬프트에 넣는다.

```
교보문고 베스트셀러 데이터 기본 전처리를 완료했습니다.
제가 실제 Notebook에서 확인한 결과는 다음과 같습니다.
- 원본 데이터 크기: [실제 shape 입력]
- 선택한 컬럼 수: [실제 개수 입력]
- 저자 결측치: [실제 개수 입력]
- 분야 결측치: [실제 개수 입력]
- 판매상품ID 중복: [실제 개수 입력]
- 판매가 자료형: [실제 dtype 입력]
- 발행일 자료형: [실제 dtype 입력]
이 결과를 바탕으로 Notebook에 넣을 짧은 Markdown 설명을 작성해 주세요.
다음 세 부분으로 작성해 주세요.
1. 무엇을 확인했는지
2. 어떤 전처리를 했는지
3. 다음 분석을 위해 어떤 상태가 되었는지
제가 제공하지 않은 숫자는 추측하지 마세요.
초보자가 작성한 것처럼 쉽고 간결하게 작성해 주세요.
```

---

> 아래 숫자는 위 셀들의 **실제 실행 결과**를 보고 적은 것이다.

### 1. 무엇을 확인했는가

교보문고 온라인 일간 베스트 Excel 파일을 불러와 구조를 확인했다.

- 원본 데이터 크기: **(986, 13)** — 986행 13컬럼
- 원본 컬럼 13개 중 분석에 필요한 **8개**만 선택
- 저자(인물) 결측치: **10개**
- 분야 결측치: **64개**
- 출판사 결측치: 11개, 발행일 결측치: 8개
- 판매상품ID 중복: **0개** (상품이 겹치지 않는다)

자료형에서 두 가지 문제를 확인했다.

- 판매가가 `'16,200'`처럼 쉼표가 붙은 **문자열(str)** 이었다.
- 발행일이 `20260701.0`처럼 **float64**였다. 결측치 8개가 섞여 있어서 pandas가 정수 대신 실수로 읽었기 때문이다.

### 2. 어떤 전처리를 했는가

- 분석에 사용할 8개 컬럼만 선택
- 컬럼명 정리 (판매상품 ID -> 판매상품ID, 인물 -> 저자, 발행(출시)일자 -> 발행일)
- 저자 결측치 10개는 "미상", 분야 결측치 64개는 "미분류"로 대체
- 판매가에서 쉼표를 제거하고 숫자형으로 변환 (`str` -> `int64`)
- 발행일을 정수로 바꾼 뒤 YYYYMMDD 형식으로 날짜 변환 (`float64` -> `datetime64`)
- 상품명, 저자, 출판사, 분야의 앞뒤 공백 제거
- 중복 데이터는 개수만 확인하고 삭제하지 않음

출판사 결측치 11개와 발행일 결측치 8개는 **일부러 처리하지 않았다.**
제목 단어 빈도 분석에는 두 컬럼이 필요하지 않기 때문에,
지금 임의로 값을 채우면 나중에 원본이 비어 있었다는 사실을 알 수 없게 된다.

### 3. 다음 분석을 위해 어떤 상태가 되었는가

- 판매가가 `int64`가 되어 평균, 합계 등 가격 통계 계산이 가능하다.
- 발행일이 `datetime64`가 되어 연도, 월 단위 분석이 가능하다.
- 상품명의 앞뒤 공백이 정리되어 단어 빈도 분석에 바로 쓸 수 있다.
- 분야의 빈 값이 "미분류"로 채워져 카테고리별 집계에서 행이 사라지지 않는다.
- `book_bestseller_clean.csv`로 저장하여 Chapter 02에서 재사용할 수 있다.

### 4. 전처리한 데이터를 살펴보고 알게 된 것

**분야** — 총 40여 개 분야가 있고 `소설` 178권(18.1%)이 가장 많았다.
그 뒤로 `인문` 100권, `어린이(초등)` 84권, `취업/수험서` 79권 순이었다.
`미분류`로 채운 64권은 전체의 6.5%로, 카테고리 분류에 쓰기에는 무시하기 어려운 양이다.

**가격** — 중앙값 16,200원, 평균 17,582원이었다.
`1.5만~2만` 구간에 445권(45.1%)이 몰려 있어 베스트셀러 가격대가 상당히 좁다.
다만 최저 2,690원부터 최고 129,600원까지 폭이 넓어, 평균만 보면 오해할 수 있다.

**발행 시기** — 2026년 발행이 541권으로 **전체의 54.9%** 였다.
베스트셀러 목록이 신간 중심으로 구성된다는 뜻이다.
반면 가장 오래된 책은 2002년 발행으로, 스테디셀러도 함께 들어 있다.

**출판사** — `민음사`가 65권으로 압도적 1위였다.
가장 싼 책 5권이 모두 `초판본` 시리즈(2,690원)인 것과 연결해 보면,
민음사 세계문학 초판본 시리즈가 여러 권 한꺼번에 순위에 올라 있기 때문으로 보인다.
즉 **출판사 순위는 브랜드 인기라기보다 시리즈 상품 수를 반영할 수 있다.**

**분야별 가격차** — `만화` 8,767원이 가장 싸고 `취업/수험서` 28,114원이 가장 비쌌다.
분야를 무시하고 전체 평균만 보면 이 차이가 가려진다.

### 5. 데이터에서 발견한 문제

가장 비싼 상품을 확인하다가 **도서가 아닌 상품**이 섞여 있는 것을 발견했다.

```
교보문고 기프트카드 10만원              분야: 상품권(기프트카드)
The Scent of Page : 디퓨저 300ML       분야: 책향
교보문고 리딩백S                        분야: 교보문고 굿즈
```

저자 상위 1위가 `교보문고`(16권)로 나온 것도 이 상품들 때문이었다.

이 파일은 "도서 목록"이 아니라 **"베스트셀러 상품 목록"** 이므로 정상적인 데이터다.
하지만 이후 분석에 영향을 준다.

- Chapter 02에서 제목 단어를 셀 때 `디퓨저`, `기프트카드` 같은 단어가 섞인다.
- Chapter 04에서 분야를 분류할 때 `상품권`, `책향`이 정답 데이터에 들어간다.

이번 Chapter에서는 **개수만 확인하고 삭제하지 않았다.**
분석 목적이 정해지기 전에 지우면, 나중에 왜 지웠는지 설명할 수 없기 때문이다.

### 참고: 주간 베스트셀러

같은 전처리를 온라인 주간 베스트에도 적용했다.

- 원본 크기: (983, 13)
- 저자 결측치 4개, 분야 결측치 49개, 출판사 5개, 발행일 3개
- 일간 판매가 평균 17,582원 / 주간 판매가 평균 18,485원
- 일간 1위와 주간 1위는 같은 책이었다.

---
# 13. 이번 Chapter에서 꼭 기억할 것

- 결측치 발견 -> 바로 삭제하지 않는다.
- 중복 발견 -> 바로 삭제하지 않는다.
- 숫자처럼 보임 -> 실제 자료형을 확인한다.
- 날짜처럼 보임 -> 실제 날짜 자료형인지 확인한다.
- AI 코드 실행 성공 -> 분석이 맞다는 뜻은 아니다.

핵심 흐름

> 확인 -> 판단 -> 전처리 -> 다시 확인

---
# 14. 확인 문제

**질문 1.** 왜 모든 원본 컬럼을 사용하지 않고 필요한 컬럼만 선택했나요?

원본에는 상품코드, 정가, 할인율, 적립율, 적립예정포인트처럼 이번 분석에 쓰지 않는 컬럼이 있었다.
이번 과정의 목표는 도서 제목으로 단어를 분석하고 카테고리를 분류하는 것이므로
상품명, 분야, 판매상품ID 같은 컬럼이 중심이 된다.
쓰지 않는 데이터를 계속 들고 있으면 전처리할 대상이 늘어나고
어떤 컬럼이 분석에 쓰이는지 헷갈리게 된다.
그래서 13개 중 8개만 남겨 데이터의 목적을 분명하게 만들었다.

**질문 2.** 판매가가 문자열이라면 어떤 문제가 발생할 수 있나요?

판매가가 `'16,200'`처럼 쉼표가 붙은 문자열이면 사람 눈에는 숫자로 보이지만
컴퓨터는 글자로 처리하기 때문에 평균, 합계, 최댓값 같은 계산을 할 수 없다.
`describe()`로 가격 분포를 보는 것도 안 되고, 가격대별로 정렬하거나 비교하는 것도 안 된다.
게다가 오류가 나지 않고 그냥 이상한 결과가 나오는 경우도 있어서 더 위험하다.
그래서 실행 전에 `dtype`을 직접 확인하는 습관이 필요하다.

**질문 3.** 결측치를 확인한 뒤 바로 모든 행을 삭제하면 안 되는 이유는 무엇인가요?

결측치가 있다고 그 행 전체가 쓸모없는 것은 아니기 때문이다.
이번 데이터에서 분야가 비어 있는 책이 64권 있었는데,
이 책들은 카테고리 분류 학습에는 쓰기 어려워도 제목 단어 빈도 분석에는 문제없이 쓸 수 있다.
만약 `dropna()`로 바로 지웠다면 986행 중 상당수를 이유 없이 잃었을 것이다.
즉 결측치를 어떻게 처리할지는 **분석 목적에 따라 달라지므로**,
먼저 어디에 얼마나 있는지 확인하고 판단해야 한다.

**질문 4.** 같은 상품명이 두 번 등장했다고 해서 바로 중복 데이터라고 판단하기 어려운 이유는 무엇인가요?

제목이 같아도 실제로는 다른 상품일 수 있기 때문이다.
개정판, 양장본과 반양장본, 세트 상품, 특별판처럼 같은 제목으로 여러 상품이 나올 수 있다.
그래서 상품을 구분하는 기준은 제목이 아니라 상품을 식별하는 값인 판매상품ID다.
실제로 이번 데이터에서 판매상품ID 중복은 0개였다.
중복은 무엇을 기준으로 볼 것인지 먼저 정한 다음, 의미를 확인하고 처리해야 한다.

**질문 5.** 인공지능이 작성한 코드가 오류 없이 실행되었다면 그것만으로 분석이 끝난 것일까요?

아니다. 코드가 도는 것과 결과가 맞는 것은 다르다.
이번 실습에서 발행일이 `20260701.0`처럼 float으로 읽혔는데,
이 상태로 그냥 문자열로 바꿔 날짜 변환을 하면 오류 없이 실행되면서
986행이 전부 변환에 실패해 값이 비어버린다.
출력값을 직접 읽어보지 않았다면 이 문제를 발견하지 못했을 것이다.
그래서 `확인 -> 판단 -> 전처리 -> 다시 확인` 순서로,
처리한 뒤에 결과를 반드시 다시 봐야 한다.

---
# 15. 제출 전 확인

Notebook을 처음부터 마지막 셀까지 다시 실행하고 아래를 점검한다.

- [x] Excel 파일이 정상적으로 로드된다.
- [x] 데이터 크기와 컬럼을 확인했다.
- [x] 필요한 컬럼만 선택했다.
- [x] 컬럼 이름을 정리했다.
- [x] 결측치를 확인하고 처리했다.
- [x] 판매가를 숫자형으로 변환했다.
- [x] 발행일을 날짜형으로 변환했다.
- [x] 문자열 앞뒤 공백을 정리했다.
- [x] 중복 데이터를 확인했다.
- [x] 최종 데이터를 다시 검증했다.
- [x] book_bestseller_clean.csv를 저장했다.
- [x] 실제 실행 결과를 바탕으로 Markdown 설명을 작성했다.
- [x] Notebook 전체를 다시 실행했을 때 오류가 없다.